In [1]:
import os
from reasoning.tools.utils import load_model_with_vllm
from reasoning.evaluator.math_grader import math_equal, extract_answer
from datasets import load_dataset
from vllm import SamplingParams
import wandb
import time
from reasoning.tools.logger import load_config, apply_config
import numpy as np
import json
import argparse
from reasoning.inference.tree import Path
from reasoning.models.model import ValueModel_qwen




In [2]:


dataset = load_dataset('HuggingFaceH4/MATH-500')
dataset = dataset['test']
# model_name = 'Qwen/Qwen2.5-7B-Instruct'
model_name = 'meta-llama/Llama-3.2-1B-Instruct'
model, tokenizer = load_model_with_vllm(model_name, task='auto', tensor_parallel_size=4, gpu_memory_utilization=0.9)
reward_model = ValueModel_qwen(device = "auto")
tokenizer.pad_token = tokenizer.eos_token 


INFO 04-12 20:51:00 config.py:510] This model supports multiple tasks: {'generate', 'score', 'embed', 'classify', 'reward'}. Defaulting to 'generate'.
INFO 04-12 20:51:00 config.py:1310] Defaulting to use mp for distributed inference
WARNING 04-12 20:51:00 arg_utils.py:1103] Chunked prefill is enabled by default for models with max_model_len > 32K. Currently, chunked prefill might not work with some features or models. If you encounter any issues, please disable chunked prefill by setting --enable-chunked-prefill=False.
INFO 04-12 20:51:00 config.py:1458] Chunked prefill is enabled with max_num_batched_tokens=2048.
INFO 04-12 20:51:00 llm_engine.py:234] Initializing an LLM engine (v0.6.6.post1) with config: model='meta-llama/Llama-3.2-1B-Instruct', speculative_config=None, tokenizer='meta-llama/Llama-3.2-1B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_se

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]


(VllmWorkerProcess pid=2410229) INFO 04-12 20:51:04 weight_utils.py:296] No model.safetensors.index.json found in remote.
(VllmWorkerProcess pid=2410230) INFO 04-12 20:51:05 weight_utils.py:296] No model.safetensors.index.json found in remote.
INFO 04-12 20:51:05 model_runner.py:1099] Loading model weights took 0.5968 GB
(VllmWorkerProcess pid=2410229) INFO 04-12 20:51:05 model_runner.py:1099] Loading model weights took 0.5968 GB
(VllmWorkerProcess pid=2410231) INFO 04-12 20:51:05 weight_utils.py:296] No model.safetensors.index.json found in remote.
(VllmWorkerProcess pid=2410230) INFO 04-12 20:51:05 model_runner.py:1099] Loading model weights took 0.5968 GB
(VllmWorkerProcess pid=2410231) INFO 04-12 20:51:05 model_runner.py:1099] Loading model weights took 0.5968 GB
(VllmWorkerProcess pid=2410231) INFO 04-12 20:51:08 worker.py:241] Memory profiling takes 3.11 seconds
(VllmWorkerProcess pid=2410231) INFO 04-12 20:51:08 worker.py:241] the current vLLM instance can use total_gpu_memory (

Capturing CUDA graph shapes:  97%|█████████▋| 34/35 [00:17<00:00,  1.91it/s]

(VllmWorkerProcess pid=2410229) INFO 04-12 20:51:31 model_runner.py:1535] Graph capturing finished in 19 secs, took 0.24 GiB
(VllmWorkerProcess pid=2410230) INFO 04-12 20:51:31 model_runner.py:1535] Graph capturing finished in 19 secs, took 0.24 GiB
(VllmWorkerProcess pid=2410231) INFO 04-12 20:51:31 model_runner.py:1535] Graph capturing finished in 19 secs, took 0.24 GiB


Capturing CUDA graph shapes: 100%|██████████| 35/35 [00:18<00:00,  1.86it/s]

INFO 04-12 20:51:31 model_runner.py:1535] Graph capturing finished in 19 secs, took 0.24 GiB
INFO 04-12 20:51:31 llm_engine.py:431] init engine (profile, create kv cache, warmup model) took 26.06 seconds


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Some weights of the model checkpoint at Qwen/Qwen2.5-Math-PRM-7B were not used when initializing Qwen2ForProcessRewardModel: ['lm_head.weight']
- This IS expected if you are initializing Qwen2ForProcessRewardModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing Qwen2ForProcessRewardModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [3]:


# inference hyperparameters
num_return_sequences = 2
max_new_tokens = 1024
temperature = 0.7
top_p = 0.9
batch_size = 10
system_prompt = 'Answer it step by step. And return the final answer in the format of \\boxed{}.'

# name of the results file
config_name =  'test'

# Also set the seed for the sampling params 
sampling_params = SamplingParams(
    temperature=temperature,
    max_tokens=max_new_tokens,
    n=num_return_sequences,
    top_p=top_p,
    stop_token_ids=[tokenizer.eos_token_id],
    skip_special_tokens = True,
    include_stop_str_in_output = False,
) # shouldn't set seed for random sampling


In [7]:
import importlib
import reasoning.inference.beam
beam_module = importlib.reload(reasoning.inference.beam)
BeamInference = beam_module.BeamInference


inference = BeamInference(model, tokenizer,sampling_params,config_name,reward_model,max_steps=20,beam_width=3)

start_time = time.time()
for i in range(0, len(dataset)):
    question = dataset['problem'][i]
    answer = dataset['solution'][i]            
    accuracy = inference.inference(question, answer)
    print("Accuracy: {}".format(accuracy))
inference.reset()
wandb.log({"accuracy": accuracy})
end_time = time.time()
print(f"parallel size: {num_return_sequences}")
print(f"generated tokens per sample in average: {np.mean(inference.num_generated_tokens) * num_return_sequences}")
wandb.log({"generated tokens per sample in average": np.mean(inference.num_generated_tokens) * num_return_sequences})
print("Time taken: {} seconds".format(end_time - start_time))

wandb.finish()

Processed prompts:   0%|          | 0/3 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts:  33%|███▎      | 1/3 [00:00<00:01,  1.33it/s, est. speed input: 403.68 toks/s, output: 465.15 toks/s]
We detected that you are passing `past_key_values` as a tuple and this is deprecated and will be removed in v4.43. Please use an appropriate `Cache` class (https://huggingface.co/docs/transformers/v4.41.3/en/internal/generation_utils#transformers.Cache)


standardized next step: Step 1: To convert the point $(0,3)$ from rectangular coordinates to polar coordinates, we need to find the radius $r$ and the angle $\theta$. The relationship between rectangular and polar coordinates is given by $x = r \cos \theta$ and $y = r \sin \theta$. Since the point is $(0,3)$, we can substitute $x=0$ and $y=3$ into these equations to get $0 = r \cos \theta$ and $3 = r \sin \theta$. We can then solve for $r$ and $\theta$ using these equations.

standardized next step: Step 1: To convert the point $(0,3)$ from rectangular coordinates to polar coordinates, we need to find the distance $r$ and the angle $\theta$ from the origin. The distance $r$ is found using the formula $r = \sqrt{x^2 + y^2}$, where $x$ and $y$ are the rectangular coordinates of the point.

standardized next step: Step 1: To convert the point $(0,3)$ from rectangular coordinates to polar coordinates, we can use the following steps:
1. Calculate the radius $r$ using the formula $r = \sqrt{

Processed prompts:  33%|███▎      | 1/3 [00:02<00:04,  2.36s/it, est. speed input: 181.24 toks/s, output: 283.71 toks/s]


standardized next step: Step 2: To find the value of $\theta$, we can use the equation $3 = r \sin \theta$, which is equivalent to $\sin \theta = \frac{3}{r}$. Using the Pythagorean identity $\sin^2 \theta + \cos^2 \theta = 1$, we can solve for $r$:

standardized next step: Step 2: To find the radius $r$, we can divide both sides of the equation $3 = r \sin \theta$ by $r$.

standardized next step: Step 2: Analyze the problem and identify the given information.
The problem requires converting the point (0,3) from rectangular coordinates to polar coordinates. We are given the rectangular coordinates (x, y) = (0, 3) and need to find the corresponding polar coordinates (r, θ).

##



...

Processed prompts:  33%|███▎      | 1/3 [00:00<00:00,  2.24it/s, est. speed input: 1299.77 toks/s, output: 431.73 toks/s]


standardized next step: Step 4: To find the value of $r$, we can substitute $x=0$ and $y=3$ into the equation $r = \sqrt{x^2 + y^2}$.

standardized next step: Step 4: We can use the fact that $x = 0$ implies $r = 0$, and we can find $\theta$ using the fact that $\tan \theta = \frac{y}{x}$, which in this case is $\tan \theta = \frac{3}{0}$. However, this is undefined, which means that the point $(0, 3)$ cannot be in polar coordinates.

standardized next step: Step 4: To find the value of $\theta$, we can use the equation $\theta = \tan^{-1}\left(\frac{y}{x}\right)$. Since $x=0$ and $y=3$, we have $\theta = \tan^{-1}(0)$.



Processed prompts:  33%|███▎      | 1/3 [00:00<00:00,  2.90it/s, est. speed input: 1806.56 toks/s, output: 392.70 toks/s]


standardized next step: Step 5: To find the value of $r$, we can substitute $x=0$ and $y=3$ into the equation $r = \sqrt{x^2 + y^2}$.

$r = \sqrt{0^2 + 3^2} = \sqrt{9} = 3$

next step is repeated！

standardized next step: Step 5: \boxed{\sqrt{0^2 + 3^2} = \sqrt{9} = 3}



ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (3,) + inhomogeneous part.

In [ ]:
import importlib
import reasoning.inference.majority
majority_module = importlib.reload(reasoning.inference.majority)
MajorityInference = majority_module.MajorityInference


inference = MajorityInference(model, tokenizer,sampling_params,config_name,reward_model,method='weighted_majority')

start_time = time.time()
for i in range(0, len(dataset), batch_size):
    questions = dataset['problem'][i:i+batch_size]
    answers = dataset['solution'][i:i+batch_size]            
    accuracy = inference.inference(system_prompt, questions, answers)
    print("Accuracy: {}".format(accuracy))
inference.reset()
wandb.log({"accuracy": accuracy})
end_time = time.time()
print(f"parallel size: {num_return_sequences}")
print(f"generated tokens per sample in average: {np.mean(inference.num_generated_tokens) * num_return_sequences}")
wandb.log({"generated tokens per sample in average": np.mean(inference.num_generated_tokens) * num_return_sequences})
print("Time taken: {} seconds".format(end_time - start_time))

wandb.finish()